In [2]:
from pathlib import Path
import re

In [3]:
# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = Path(".")  # folder where your notebook is running
SRC_RUNS = [
    ("calibration data run 1", "fixed calibration data/run 1 calibration data edit"),
    ("calibration data run 2", "fixed calibration data/run 2 calibration data edit"),
]
MAX_LINES = 500

# read numbers in name until they stop
LEADING_NUMBER_RE = re.compile(r"^(\d+(?:\.\d+)?)")

def first_n_nonempty_lines(path: Path, n: int) -> list[str]:
    """
    Read file as text, return up to first n non-empty lines (stripped of trailing newlines).
    """
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"[SKIP] Could not read {path} ({e})")
        return []

    lines = []
    for line in text.splitlines():
        s = line.strip()
        if not s:
            continue
        lines.append(line.rstrip("\n\r"))
        if len(lines) >= n:
            break
    return lines

def process_run(src_rel: str, dst_rel: str):
    src_dir = BASE_DIR / src_rel
    dst_dir = BASE_DIR / dst_rel
    dst_dir.mkdir(parents=True, exist_ok=True)

    if not src_dir.exists():
        print(f"[WARN] Source folder not found: {src_dir}")
        return

    total = 0
    ignored_non_numeric = 0
    ignored_empty = 0
    written = 0

    # iterate files (not subfolders)
    for p in sorted(src_dir.iterdir()):
        if p.is_dir():
            continue
        total += 1

        m = LEADING_NUMBER_RE.match(p.name)
        if not m:
            ignored_non_numeric += 1
            continue

        # Read up to 500 non-empty lines
        lines = first_n_nonempty_lines(p, MAX_LINES)
        if not lines:
            ignored_empty += 1
            continue

        # output name: original stem + "_edit.txt"
        # (keeps the "…ns" part exactly as-is, and adds "_edit")
        out_name = f"{p.stem}_edit.txt"
        out_path = dst_dir / out_name

        out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        written += 1

    print(f"\n=== {src_rel} -> {dst_rel} ===")
    print(f"Files seen:                {total}")
    print(f"Ignored (no leading nums): {ignored_non_numeric}")
    print(f"Ignored (empty/no lines):  {ignored_empty}")
    print(f"Written:                   {written}")

In [4]:
# ----------------------------
# RUN
# ----------------------------
for src_rel, dst_rel in SRC_RUNS:
    process_run(src_rel, dst_rel)


=== calibration data run 1 -> fixed calibration data/run 1 calibration data edit ===
Files seen:                41
Ignored (no leading nums): 1
Ignored (empty/no lines):  1
Written:                   39

=== calibration data run 2 -> fixed calibration data/run 2 calibration data edit ===
Files seen:                33
Ignored (no leading nums): 1
Ignored (empty/no lines):  5
Written:                   27
